<a href="https://vigneashpandiyan.github.io/publications/Codes/" target="_blank" rel="noopener noreferrer">
  <img src="https://vigneashpandiyan.github.io/images/Link.png"
       style="max-width: 800px; width: 100%; height: auto;">
</a>

# Building Feedforward Neural Network Using Pytorch

# Import libraries

In [ ]:
#required libraries

import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.colors
import time

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, log_loss
from tqdm import tqdm_notebook

from IPython.display import HTML
import warnings
from sklearn.preprocessing import OneHotEncoder
from sklearn.datasets import make_blobs

import torch
import torch.nn.functional as F

warnings.filterwarnings('ignore')

In [ ]:
#setting the manual seed
torch.manual_seed(0)

In [ ]:
#generate custom color map

my_cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", ["red","yellow","green"])

# Generate Dataset Forward pass with Native Pytorch functions

In [ ]:
#generate data using make_blobs function from sklearn.
#centers = 4 indicates different types of classes

data, labels = make_blobs(n_samples=1000, centers=4, n_features=2, random_state=0)
print(data.shape, labels.shape)

In [ ]:
plt.scatter(data[:,0], data[:,1], c=labels, cmap=my_cmap)
plt.show()

In [ ]:
#splitting the data into train and test

X_train, X_val, Y_train, Y_val = train_test_split(data, labels, stratify=labels, random_state=0)
print(X_train.shape, X_val.shape, labels.shape)

# Using torch tensors and autograd

In [ ]:
#converting the numpy array to torch tensors
X_train, Y_train, X_val, Y_val = map(torch.tensor, (X_train, Y_train, X_val, Y_val))
print(X_train.shape, Y_train.shape)

In [ ]:
#function for computing forward pass in the network
def model(x):
    a1 = torch.matmul(x, weights1) + bias1 # (N, 2) x (2, 2) -> (N, 2)
    h1 = a1.sigmoid() # (N, 2)
    a2 = torch.matmul(h1, weights2) + bias2 # (N, 2) x (2, 4) -> (N, 4)
    h2 = a2.exp()/a2.exp().sum(-1).unsqueeze(-1) # (N, 4)
    return h2

In [ ]:
#demo of how we calculate the cross entropy loss

y_hat = torch.tensor([[0.1, 0.2, 0.3, 0.4], [0.8, 0.1, 0.05, 0.05]])
y = torch.tensor([2, 0])
(-y_hat[range(y_hat.shape[0]), y].log()).mean().item()
(torch.argmax(y_hat, dim=1) == y).float().mean().item()

In [ ]:
#function to calculate loss of a function.
#y_hat -> predicted & y -> actual
def loss_fn(y_hat, y):
    return -(y_hat[range(y.shape[0]), y].log()).mean()

In [ ]:
#function to calculate accuracy of model
def accuracy(y_hat, y):
    pred = torch.argmax(y_hat, dim=1)
    return (pred == y).float().mean()

# Forward pass

In [ ]:
#set the seed
torch.manual_seed(0)

#initialize the weights and biases using He Initialization
weights1 = torch.randn(2, 2) / math.sqrt(2)
weights1.requires_grad_()
bias1 = torch.zeros(2, requires_grad=True)

weights2 = torch.randn(2, 4) / math.sqrt(2)
weights2.requires_grad_()
bias2 = torch.zeros(4, requires_grad=True)

#set the parameters for training the model
learning_rate = 0.2
epochs = 10000

X_train = X_train.float()
Y_train = Y_train.long()
X_val = X_val.float()
Y_val = Y_val.long()

loss_arr = []
acc_arr = []
val_acc_arr = []

#training the network
for epoch in range(epochs):

    y_hat = model(X_train)  #compute the predicted distribution

# Print the shapes
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of y_hat: {y_hat.shape}")

# Training Model

In [ ]:
#set the seed
torch.manual_seed(0)

#initialize the weights and biases using He Initialization
weights1 = torch.randn(2, 2) / math.sqrt(2)
weights1.requires_grad_()
bias1 = torch.zeros(2, requires_grad=True)

weights2 = torch.randn(2, 4) / math.sqrt(2)
weights2.requires_grad_()
bias2 = torch.zeros(4, requires_grad=True)

#set the parameters for training the model
learning_rate = 0.2
epochs = 1000

X_train = X_train.float()
Y_train = Y_train.long()
X_val = X_val.float()
Y_val = Y_val.long()

loss_arr = []
acc_arr = []
val_acc_arr = []
val_loss_arr = [] # New array to store validation loss

#training the network
for epoch in range(epochs):
    y_hat = model(X_train)  #compute the predicted distribution
    loss = loss_fn(y_hat, Y_train) #compute the loss of the network
    loss.backward() #backpropagate the gradients
    loss_arr.append(loss.item())
    acc_arr.append(accuracy(y_hat, Y_train))

    with torch.no_grad(): #update the weights and biases
        y_val_hat = model(X_val)
        val_loss = loss_fn(y_val_hat, Y_val)
        val_loss_arr.append(val_loss.item())
        val_acc_arr.append(accuracy(y_val_hat,Y_val))

        weights1 -= weights1.grad * learning_rate
        bias1 -= bias1.grad * learning_rate
        weights2 -= weights2.grad * learning_rate
        bias2 -= bias2.grad * learning_rate
        weights1.grad.zero_()
        bias1.grad.zero_()
        weights2.grad.zero_()
        bias2.grad.zero_()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch + 1}/{epochs}, Train Loss: {loss_arr[-1]:.4f}, Train Acc: {acc_arr[-1]:.4f}, Val Loss: {val_loss_arr[-1]:.4f}, Val Acc: {val_acc_arr[-1]:.4f}')

# Plotting Loss
plt.figure(figsize=(10, 5))
plt.plot(loss_arr, 'r-', label='Train Loss')
plt.plot(val_loss_arr, 'b-', label='Validation Loss')
plt.title("Loss plot - Using tensors and autograd")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(loc='best')
plt.show()

# Plotting Accuracy
plt.figure(figsize=(10, 5))
plt.plot(acc_arr, 'r-', label='Train Accuracy')
plt.plot(val_acc_arr, 'b-', label='Validation Accuracy')
plt.title("Accuracy plot - Using tensors and autograd")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(loc='best')
plt.show()

print('Loss before training', loss_arr[0])
print('Loss after training', loss_arr[-1])

## Using NN.Parameter

In [ ]:
import torch.nn as nn

In [ ]:
class FirstNetwork(nn.Module):

    def __init__(self):
        super().__init__()
        torch.manual_seed(0)
        #wrap all the weights and biases inside nn.parameter()
        self.weights1 = nn.Parameter(torch.randn(2, 2) / math.sqrt(2))
        self.bias1 = nn.Parameter(torch.zeros(2))
        self.weights2 = nn.Parameter(torch.randn(2, 4) / math.sqrt(2))
        self.bias2 = nn.Parameter(torch.zeros(4))

    def forward(self, X):
        a1 = torch.matmul(X, self.weights1) + self.bias1
        h1 = a1.sigmoid()
        a2 = torch.matmul(h1, self.weights2) + self.bias2
        h2 = a2.exp()/a2.exp().sum(-1).unsqueeze(-1)
        return h2

In [ ]:
def fit(epochs = 10000, learning_rate = 0.2, title = ""):
    loss_arr = []
    acc_arr = []
    for epoch in range(epochs):
        y_hat = model(X_train) #forward pass
        loss = F.cross_entropy(y_hat, Y_train) #loss calculation
        loss_arr.append(loss.item())
        acc_arr.append(accuracy(y_hat, Y_train))
        loss.backward() #backpropagation
        with torch.no_grad():
            #updating the parameters
            for param in model.parameters():
                param -= learning_rate * param.grad
            model.zero_grad() #setting the gradients to zero

    # Plotting Loss
    plt.figure(figsize=(10, 5))
    plt.plot(loss_arr, 'r-', label='loss')
    plt.title(title + " - Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(loc='best')
    plt.show()

    # Plotting Accuracy
    plt.figure(figsize=(10, 5))
    plt.plot(acc_arr, 'b-', label='train accuracy')
    plt.title(title + " - Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend(loc='best')
    plt.show()

    print('Loss before training', loss_arr[0])
    print('Loss after training', loss_arr[-1])

In [ ]:
#we first have to instantiate our model
model = FirstNetwork()

#call fit method
fit(10000,0.2,"Loss plot - nn.Parameter & nn.Module")

## Using NN.Linear

In [ ]:
class FirstNetwork_v1(nn.Module):

    def __init__(self):
        super().__init__()
        torch.manual_seed(0)
        self.lin1 = nn.Linear(2, 2) #automatically defines weights and biases
        self.lin2 = nn.Linear(2, 4)

    def forward(self, X):
        a1 = self.lin1(X) #computes the dot product and adds bias
        h1 = a1.sigmoid()
        a2 = self.lin2(h1) #computes dot product and adds bias
        h2 = a2.exp()/a2.exp().sum(-1).unsqueeze(-1)
        return h2

In [ ]:
model = FirstNetwork_v1()
fit(10000,0.2,"Loss plot - nn.Linear")

## Using NN.Sequential and torch.Optim

In [ ]:
class FirstNetwork_v2(nn.Module):

    def __init__(self):
        super().__init__()
        torch.manual_seed(0)
        self.net = nn.Sequential( #sequential operation
            nn.Linear(2, 2),
            nn.Sigmoid(),
            nn.Linear(2, 4),
            nn.Softmax())

    def forward(self, X):
        return self.net(X)

In [ ]:
model = FirstNetwork_v2() #object

def fit_v2(x, y, model, opt, loss_fn, epochs = 10000):
    """Generic function for training a model """
    for epoch in range(epochs):
        loss = loss_fn(model(x), y)

        loss.backward()
        opt.step()
        opt.zero_grad()

    return loss.item()

In [ ]:
#define loss
loss_fn = F.cross_entropy
#define optimizer
opt = torch.optim.SGD(model.parameters(), lr=0.2)

#training model
fit_v2(X_train, Y_train, model, opt, loss_fn)

## Refactor and Centralize Training Process

We need to instantiate the `FirstNetwork_v2` model, define the loss function and optimizer, and then call the `fit_v3` function to train the model.



In [ ]:
import torch.nn.functional as F
from torch import optim
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

train_dataset = TensorDataset(X_train, Y_train)
val_dataset = TensorDataset(X_val, Y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

# Re-define FirstNetwork_v2 class to ensure it's available in this context
class FirstNetwork_v2(nn.Module):

    def __init__(self):
        super().__init__()
        torch.manual_seed(0)
        self.net = nn.Sequential( #sequential operation
            nn.Linear(2, 2),
            nn.Sigmoid(),
            nn.Linear(2, 4),
            nn.Softmax(dim=1))

    def forward(self, X):
        return self.net(X)

# Re-initialize the model (or instantiate for the first time in this section)
model = FirstNetwork_v2()

# Define loss function and optimizer
loss_fn = F.cross_entropy
opt = torch.optim.SGD(model.parameters(), lr=0.2)

# Re-define accuracy function if it's not globally available
def accuracy(y_hat, y):
    pred = torch.argmax(y_hat, dim=1)
    return (pred == y).float().mean()

# Re-define fit_v3 function if it's not globally available
def fit_v3(model, opt, loss_fn, train_loader, val_loader, epochs=100):
    train_loss_arr = []
    train_acc_arr = []
    val_loss_arr = []
    val_acc_arr = []

    print("Starting training with batches...")

    for epoch in range(epochs):
        # Training phase
        model.train() # Set model to training mode
        total_train_loss = 0
        total_train_correct = 0
        total_train_samples = 0

        for X_batch, Y_batch in train_loader:
            # Forward pass
            y_hat = model(X_batch)
            loss = loss_fn(y_hat, Y_batch)

            # Backward pass and optimization
            loss.backward()
            opt.step()
            opt.zero_grad()

            # Accumulate training metrics
            total_train_loss += loss.item() * X_batch.size(0) # Multiply by batch size for correct average
            total_train_correct += (torch.argmax(y_hat, dim=1) == Y_batch).sum().item()
            total_train_samples += X_batch.size(0)

        avg_train_loss = total_train_loss / total_train_samples
        avg_train_acc = total_train_correct / total_train_samples
        train_loss_arr.append(avg_train_loss)
        train_acc_arr.append(avg_train_acc)

        # Validation phase
        model.eval() # Set model to evaluation mode
        total_val_loss = 0
        total_val_correct = 0
        total_val_samples = 0

        with torch.no_grad(): # Disable gradient calculations for validation
            for X_val_batch, Y_val_batch in val_loader:
                y_val_hat = model(X_val_batch)
                val_loss = loss_fn(y_val_hat, Y_val_batch)

                # Accumulate validation metrics
                total_val_loss += val_loss.item() * X_val_batch.size(0)
                total_val_correct += (torch.argmax(y_val_hat, dim=1) == Y_val_batch).sum().item()
                total_val_samples += X_val_batch.size(0)

        avg_val_loss = total_val_loss / total_val_samples
        avg_val_acc = total_val_correct / total_val_samples
        val_loss_arr.append(avg_val_loss)
        val_acc_arr.append(avg_val_acc)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_acc:.4f} | Val Loss: {avg_val_loss:.4f}, Val Acc: {avg_val_acc:.4f}")

    # Plotting results
    plt.figure(figsize=(10, 5))
    plt.plot(train_loss_arr, 'r-', label='Train Loss')
    plt.plot(val_loss_arr, 'b-', label='Validation Loss')
    plt.title("Loss over Epochs (Batched Training)")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(train_acc_arr, 'r-', label='Train Accuracy')
    plt.plot(val_acc_arr, 'b-', label='Validation Accuracy')
    plt.title("Accuracy over Epochs (Batched Training)")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    print(f'Final Train Loss: {train_loss_arr[-1]:.4f}, Final Train Accuracy: {train_acc_arr[-1]:.4f}')
    print(f'Final Validation Loss: {val_loss_arr[-1]:.4f}, Final Validation Accuracy: {val_acc_arr[-1]:.4f}')
    return train_loss_arr, train_acc_arr, val_loss_arr, val_acc_arr

# Call the new training function
train_losses, train_accuracies, val_losses, val_accuracies = fit_v3(model, opt, loss_fn, train_loader, val_loader, epochs=100)